In [274]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
import webbrowser
import os
from datetime import datetime
import pytz
import random



In [275]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Shubham\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [276]:
apps_df = pd.read_csv("Play Store Data.csv")
review_df = pd.read_csv("User Reviews.csv")

In [277]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [278]:
apps_df.columns

Index(['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type',
       'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver',
       'Android Ver'],
      dtype='object')

In [279]:
apps_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  object 
 1   Category        10841 non-null  object 
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  object 
 4   Size            10841 non-null  object 
 5   Installs        10841 non-null  object 
 6   Type            10840 non-null  object 
 7   Price           10841 non-null  object 
 8   Content Rating  10840 non-null  object 
 9   Genres          10841 non-null  object 
 10  Last Updated    10841 non-null  object 
 11  Current Ver     10833 non-null  object 
 12  Android Ver     10838 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.1+ MB


In [280]:
apps_df.describe()

,Rating
count,9367.000000
mean,4.193338
std,0.537431
min,1.000000
25%,4.000000
50%,4.300000
75%,4.500000
max,19.000000


In [281]:
apps_df.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
10836    False
10837    False
10838    False
10839    False
10840    False
Length: 10841, dtype: bool

In [282]:
apps_df.isnull()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10836,False,False,False,False,False,False,False,False,False,False,False,False,False
10837,False,False,False,False,False,False,False,False,False,False,False,False,False
10838,False,False,True,False,False,False,False,False,False,False,False,False,False
10839,False,False,False,False,False,False,False,False,False,False,False,False,False


In [283]:
review_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [284]:
review_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64295 entries, 0 to 64294
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   App                     64295 non-null  object 
 1   Translated_Review       37427 non-null  object 
 2   Sentiment               37432 non-null  object 
 3   Sentiment_Polarity      37432 non-null  float64
 4   Sentiment_Subjectivity  37432 non-null  float64
dtypes: float64(2), object(3)
memory usage: 2.5+ MB


In [285]:
review_df.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
64290     True
64291     True
64292     True
64293     True
64294     True
Length: 64295, dtype: bool

In [286]:
 # data cleaning
apps_df = apps_df.dropna(subset=["Rating"])
for column in apps_df.columns:
    apps_df[column].fillna(apps_df[column].mode()[0], inplace=True)
apps_df.drop_duplicates(inplace=True)
apps_df=apps_df[apps_df['Rating']<=5]
review_df.dropna(subset=["Translated_Review"], inplace=True)

C:\Windows\Temp\ipykernel_24968\4280260988.py:4: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





In [287]:
# convert the instails columns to numeric by removing commas and +
apps_df['Installs'] = apps_df['Installs'].str.replace(',','').str.replace('+','').astype(int)

# convert price columns to numeric after remocing $
apps_df['Price'] = apps_df['Price'].str.replace('$','').astype(float)


In [288]:
apps_df.dtypes

App                object
Category           object
Rating            float64
Reviews            object
Size               object
Installs            int64
Type               object
Price             float64
Content Rating     object
Genres             object
Last Updated       object
Current Ver        object
Android Ver        object
dtype: object

In [289]:
merged_df = pd.merge(apps_df,review_df,on='App',how='inner')
merged_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,A kid's excessive ads. The types ads allowed a...,Negative,-0.250,1.000000
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,It bad >:(,Negative,-0.725,0.833333
2,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,like,Neutral,0.000,0.000000
3,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,I love colors inspyering,Positive,0.500,0.600000
4,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,I hate,Negative,-0.800,0.900000


In [290]:
merged_df.columns

Index(['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type',
       'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver',
       'Android Ver', 'Translated_Review', 'Sentiment', 'Sentiment_Polarity',
       'Sentiment_Subjectivity'],
      dtype='object')

In [291]:
# creating the column countrr for task 2
countries = ['United States', 'India', 'Brazil', 'Russia', 'Japan', 'Germany', 'United Kingdom', 'France', 'Italy', 'Canada', 'Australia', 'Spain', 'Mexico', 'Netherlands', 'South Korea', 'Turkey', 'Saudi Arabia', 'Indonesia', 'Sweden', 'Switzerland']

In [292]:
merged_df["Country"] = np.random.choice(countries, size=len(merged_df))
merged_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity,Country
0,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,A kid's excessive ads. The types ads allowed a...,Negative,-0.250,1.000000,Mexico
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,It bad >:(,Negative,-0.725,0.833333,France
2,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,like,Neutral,0.000,0.000000,Germany
3,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,I love colors inspyering,Positive,0.500,0.600000,United Kingdom
4,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up,I hate,Negative,-0.800,0.900000,Canada


In [293]:
merged_df.isnull()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity,Country
0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59119,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
59120,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
59121,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
59122,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [294]:
merged_df.duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
59119    False
59120    False
59121    False
59122    False
59123    False
Length: 59124, dtype: bool

In [295]:
merged_df.dtypes

App                        object
Category                   object
Rating                    float64
Reviews                    object
Size                       object
Installs                    int64
Type                       object
Price                     float64
Content Rating             object
Genres                     object
Last Updated               object
Current Ver                object
Android Ver                object
Translated_Review          object
Sentiment                  object
Sentiment_Polarity        float64
Sentiment_Subjectivity    float64
Country                    object
dtype: object

In [296]:
merged_df['Reviews'] = pd.to_numeric(merged_df['Reviews'], errors='coerce')

In [297]:
merged_df['Last Updated'] = pd.to_datetime(merged_df['Last Updated'], errors='coerce')

In [298]:
merged_df["Revenue"] = merged_df["Installs"] * merged_df["Price"]

In [300]:
# data  transformation  
def convert_size(size):
    if 'M' in size:
        return float(size.replace('M',''))
    elif 'K' in size:
        return float(size.replace('K','')) / 1024
    else:
        return np.nan
apps_df['Size'] = apps_df['Size'].apply(convert_size)

In [301]:
merged_df['Size'] = pd.to_numeric(merged_df['Size'], errors='coerce')

In [302]:
# lagromithic 
apps_df['Reviews'] = apps_df['Reviews'].astype(int)
apps_df['Log_Installs']=np.log1p(apps_df["Installs"])
apps_df['Log_Reviews']=np.log1p(apps_df['Reviews'])

In [303]:
def rating_group(rating):
    if rating >= 4:
        return 'Top rated app'
    elif rating >=3:
        return 'Above average app'
    elif rating >=2:
        return 'Average app'
    else:
        return 'Below Average'
apps_df['Rating_Group'] = apps_df['Rating'].apply(rating_group)


In [304]:
# revenue column
apps_df['Revenue'] = apps_df['Price']*apps_df['Installs']


In [305]:
sia = SentimentIntensityAnalyzer()

In [306]:
# Polarity score in SIA
# Postive,Negative,Neutral and Coumpund: -1 - very negative ; +1 -very positive  

In [307]:
# review_df = "This app is amazing! I Love the New feature"
# sentiment_scores = sia.polarity_scores(review_df)
# print(sentiment_scores)

In [308]:
# review_df = 'This app is terrible! I hate the new update and it keeps crashing. '
# sentiment_scores = sia.polarity_scores(review_df)
# print(sentiment_scores) 

In [309]:
review_df['Sentiment_Score'] = review_df['Translated_Review'].apply(lambda x: sia.polarity_scores(x)['compound'])

In [310]:
review_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity,Sentiment_Score
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333,0.9531
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462,0.6597
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000,0.6249
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000,0.6369
5,10 Best Foods for You,Best way,Positive,1.00,0.300000,0.6369


In [311]:
apps_df['Last Updated'] = pd.to_datetime(apps_df['Last Updated'],errors='coerce')

In [312]:
apps_df['Year'] = apps_df['Last Updated'].dt.year

In [313]:
apps_df['Category'].unique()

array(['ART_AND_DESIGN', 'AUTO_AND_VEHICLES', 'BEAUTY',
       'BOOKS_AND_REFERENCE', 'BUSINESS', 'COMICS', 'COMMUNICATION',
       'DATING', 'EDUCATION', 'ENTERTAINMENT', 'EVENTS', 'FINANCE',
       'FOOD_AND_DRINK', 'HEALTH_AND_FITNESS', 'HOUSE_AND_HOME',
       'LIBRARIES_AND_DEMO', 'LIFESTYLE', 'GAME', 'FAMILY', 'MEDICAL',
       'SOCIAL', 'SHOPPING', 'PHOTOGRAPHY', 'SPORTS', 'TRAVEL_AND_LOCAL',
       'TOOLS', 'PERSONALIZATION', 'PRODUCTIVITY', 'PARENTING', 'WEATHER',
       'VIDEO_PLAYERS', 'NEWS_AND_MAGAZINES', 'MAPS_AND_NAVIGATION'],
      dtype=object)

In [314]:
apps_df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Log_Installs,Log_Reviews,Rating_Group,Revenue,Year
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19.0,10000,Free,0.0,Everyone,Art & Design,2018-01-07,1.0.0,4.0.3 and up,9.210440,5.075174,Top rated app,0.0,2018
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,Free,0.0,Everyone,Art & Design;Pretend Play,2018-01-15,2.0.0,4.0.3 and up,13.122365,6.875232,Above average app,0.0,2018
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7,5000000,Free,0.0,Everyone,Art & Design,2018-08-01,1.2.4,4.0.3 and up,15.424949,11.379520,Top rated app,0.0,2018
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,50000000,Free,0.0,Teen,Art & Design,2018-06-08,Varies with device,4.2 and up,17.727534,12.281389,Top rated app,0.0,2018
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,100000,Free,0.0,Everyone,Art & Design;Creativity,2018-06-20,1.1,4.4 and up,11.512935,6.875232,Top rated app,0.0,2018


In [315]:
fig=px.bar(x=["A","B","C"],y=[1,3,2],title="Sample Bar Chart")
fig.show()

In [316]:
fig.write_html('Intractive_plot.html')

In [317]:
# static visualization : fixed images or plots , non intractive
# intractive Visulization : 

In [318]:
html_files_path="./dasboard/"
if not os.path.exists(html_files_path):
    os.makedirs(html_files_path)

In [319]:
plot_containers=""  

In [320]:
# Save each Plotly figure to an HTML file
def save_plot_as_html(fig, filename, insight):
    global plot_containers
    filepath = os.path.join(html_files_path, filename)
    html_content = pio.to_html(fig, full_html=False, include_plotlyjs='inline')
    # Append the plot and its insight to plot_containers
    plot_containers += f"""
    <div class="plot-container" id="{filename}" onclick="openPlot('{filename}')">
        <div class="plot">{html_content}</div>
        <div class="insights">{insight}</div>
    </div>
    """
    fig.write_html(filepath, full_html=False, include_plotlyjs='inline')


In [321]:
plot_width = 500
plot_height = 400
plot_bg_color = 'black'
text_color = 'white'
title_font = {'size': 16}
axis_font = {'size': 12}


In [322]:
#Figure 1
category_counts=apps_df['Category'].value_counts().nlargest(10)
fig1=px.bar(
    x=category_counts.index,
    y=category_counts.values,
    labels={'x':'Category','y':'Count'},
    title='Top Categories on Play Store',
    color=category_counts.index,
    color_discrete_sequence=px.colors.sequential.Plasma,
    width=plot_width,
    height=plot_height
)
fig1.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10,r=10,t=30,b=10)
)
fig1.update_traces(marker=dict(line=dict(color='white',width=1)))
save_plot_as_html(fig1,"Category Graph 1.html","The top categories on the Play Store are dominated by tools, entertainment, and productivity apps")
            

In [323]:
#Figure 2
type_counts=apps_df['Type'].value_counts()
fig2=px.pie(
    values=type_counts.values,
    names=type_counts.index,
    title='App Type Distribution',
    color_discrete_sequence=px.colors.sequential.RdBu,
    width=plot_width,
    height=plot_height
)
fig2.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(line=dict(color='white',width=1)))
save_plot_as_html(fig2,"Type Graph 2.html","Most apps on the Playstore are free, indicating a strategy to attract users first and monetize through ads or in app purchases")

In [324]:
#Figure 3
fig3=px.histogram(
    apps_df,
    x='Rating',
    nbins=20,
    title='Rating Distribution',
    color_discrete_sequence=['#636EFA'],
    width=plot_width,
    height=plot_height
)
fig3.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig1.update_traces(marker=dict(line=dict(color='white',width=1)))
save_plot_as_html(fig3,"Rating Graph 3.html","Ratings are skewed towards higher values, suggesting that most apps are rated favorably by users")

In [325]:
#Figure 4
sentiment_counts=review_df['Sentiment_Score'].value_counts()
fig4=px.bar(
    x=sentiment_counts.index,
    y=sentiment_counts.values,
    labels={'x':'Sentiment Score','y':'Count'},
    title='Sentiment Distribution',
    color=sentiment_counts.index,
    color_discrete_sequence=px.colors.sequential.RdPu,
    width=plot_width,
    height=plot_height
)
fig4.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10,r=10,t=30,b=10)
)
fig4.update_traces(marker=dict(line=dict(color='white',width=1)))
save_plot_as_html(fig4,"Sentiment Graph 4.html","Sentiments in reviews show a mix of positive and negative feedback, with a slight lean towards positive sentiments")

In [326]:
#Figure 5
installs_by_category=apps_df.groupby('Category')['Installs'].sum().nlargest(10)
fig5=px.bar(
    x=installs_by_category.index,
    y=installs_by_category.values,
    orientation='h',
    labels={'x':'Installs','y':'Category'},
    title='Installs by Category',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.Blues,
    width=plot_width,
    height=plot_height
)
fig5.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10,r=10,t=30,b=10)
)
fig5.update_traces(marker=dict(line=dict(color='white',width=1)))
save_plot_as_html(fig5,"Installs Graph 5.html","The categories with the most installs are social and communication apps, reflecting their broad appeal and daily usage")

In [327]:
# Updates Per Year Plot
updates_per_year = apps_df['Last Updated'].dt.year.value_counts().sort_index()
fig6 = px.line(
    x=updates_per_year.index,
    y=updates_per_year.values,
    labels={'x': 'Year', 'y': 'Number of Updates'},
    title='Number of Updates Over the Years',
    color_discrete_sequence=['#AB63FA'],
    width=plot_width,
    height=plot_height
)
fig6.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
save_plot_as_html(fig6, "Updates Graph 6.html", "Updates have been increasing over the years, showing that developers are actively maintaining and improving their apps.")

In [328]:
#Figure 7
revenue_by_category=apps_df.groupby('Category')['Revenue'].sum().nlargest(10)
fig7=px.bar(
    x=installs_by_category.index,
    y=installs_by_category.values,
    labels={'x':'Category','y':'Revenue'},
    title='Revenue by Category',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.Greens,
    width=plot_width,
    height=plot_height
)
fig7.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10,r=10,t=30,b=10)
)
fig7.update_traces(marker=dict(line=dict(color='white',width=1)))
save_plot_as_html(fig7,"Revenue Graph 7.html","Categories such as Business and Productivity lead in revenue generation, indicating their monetization potential")

In [329]:
#Figure 8
genre_counts=apps_df['Genres'].str.split(';',expand=True).stack().value_counts().nlargest(10)
fig8=px.bar(
    x=genre_counts.index,
    y=genre_counts.values,
    labels={'x':'Genre','y':'Count'},
    title='Top Genres',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.OrRd,
    width=plot_width,
    height=plot_height
)
fig8.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10,r=10,t=30,b=10)
)
fig8.update_traces(marker=dict(line=dict(color='white',width=1)))
save_plot_as_html(fig8,"Genre Graph 8.html","Action and Casual genres are the most common, reflecting users' preference for engaging and easy-to-play games")

In [330]:
#Figure 9
fig9=px.scatter(
    apps_df,
    x='Last Updated',
    y='Rating',
    color='Type',
    title='Impact of Last Update on Rating',
    color_discrete_sequence=px.colors.qualitative.Vivid,
    width=plot_width,
    height=plot_height
)
fig9.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig9.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig9,"Update Graph 9.html","The Scatter Plot shows a weak correlation between the last update and ratings, suggesting that more frequent updates dont always result in better ratings.")

In [331]:
#Figure 10
fig10=px.box(
    apps_df,
    x='Type',
    y='Rating',
    color='Type',
    title='Rating for Paid vs Free Apps',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    width=plot_width,
    height=plot_height
)
fig10.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10,r=10,t=30,b=10)
)
#fig10.update_traces(marker=dict(pattern=dict(line=dict(color='white',width=1))))
save_plot_as_html(fig10,"Paid Free Graph 10.html","Paid apps generally have higher ratings compared to free apps, suggesting that users expect higher quality from apps they pay for")

In [346]:
# task 1 : Bubble chart : Relation btw app size and rating
# applying the filter
Categories = ['GAME','BEAUTY','BUSINESS','COMICS','COMMUNICATION','DATING','ENTERTAINMENT','SOCIAL','EVENTS']
condition_df = merged_df[(merged_df['Rating'] > 3.5) &  (merged_df['Category'].isin(Categories)) & (merged_df['Reviews'] > 500) & (~merged_df['App'].str.contains('S',case = False, na=False)) & (merged_df['Sentiment_Subjectivity'] > 0.5) & (merged_df['Installs'] > 50000)].copy()
category_translation =  {'BEAUTY':"सौंदर्य",'BUSINESS':'வணிகம்','DATING':'Verabreden'}
condition_df['Translated_Category'] = condition_df['Category'].replace(category_translation)
#time restriction 
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist)
#Checking time btw 5pm and 7pm
if 17 <= current_time.hour < 19:
    show_graph = True
else:
    show_graph = False
if show_graph and not condition_df.empty:
    fig11 = px.scatter(
        condition_df,
        x='Size',
        y='Rating',
        size='Installs',
        color='Translated_Category',
        color_discrete_map={"GAME": "pink"},
        title='App Size vs Rating (Filtered)',
        color_discrete_sequence=px.colors.qualitative.Set2,
        width=plot_width,
        height=plot_height
    )
    
    fig11.update_layout(
        plot_bgcolor='black',
        paper_bgcolor='black',
        font_color='white',
        title_font={'size':16},
        xaxis=dict(title_font={'size':12}),
        yaxis=dict(title_font={'size':12}),
        margin=dict(l=10,r=10,t=30,b=10)
    )
    fig11.update_traces(marker=dict(line=dict(color='white',width=1)))
    save_plot_as_html(fig11, "Size Rating Graph 11.html", "This bubble chart shows the relationship between app size and rating for a filtered subset of apps. Larger bubbles indicate more installs, and colors represent different categories.")
    fig11.show()
else:
    print('Bubble chart is only available between 5 PM and 7 PM IST.')

Bubble chart is only available between 5 PM and 7 PM IST.


In [347]:
# task 2 : filter and map
filter_map_df = merged_df[~merged_df['Category'].str.startswith(('A', 'C', 'G', 'S'), na = False)] 
#Category by Installs
Top_categories = (filter_map_df.groupby('Category')['Installs'].sum().sort_values(ascending=False).head(5).index)
filter_map_df = filter_map_df[filter_map_df['Category'].isin(Top_categories)]
#Country by installs
country_installs = filter_map_df.groupby(['Country','Category'])['Installs'].sum().reset_index()
# highligh > 1M installs
country_installs['Installs_highlight'] = country_installs['Installs'].apply(lambda x: "Above 1M" if x > 1000000 else "Below 1M")
#time restriction(6PM-8PM)
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist)
if 18 <= current_time.hour < 20 :
    show_graph = True
else:
    show_graph = False
if show_graph and not country_installs.empty:
    fig12 = px.choropleth(
        country_installs,
        locations = 'Country',
        locationmode = 'ISO-3',
        color='Installs_highlight',
        hover_name='Category',
        hover_data=['Installs'],    
        title="Global Installs by App Category",
        color_discrete_map={'Above 1M': 'blue', 'Below 1M': 'lightgray'},
        width=plot_width,
        height=plot_height
    )
    fig12.update_layout(
        plot_bgcolor=plot_bg_color,
        paper_bgcolor=plot_bg_color,
        font_color=text_color,
         margin=dict(l=10,r=10,t=30,b=10)
    )   
    fig12.update_traces(marker=dict(line=dict(color='white',width=1)))
    save_plot_as_html(fig12, "Global Installs Graph 12.html", "This choropleth map visualizes the total installs by country for the top app categories. Countries with more installs are highlighted in brighter colors.")
    fig12.show()
else:
    print('Choropleth map is only available between 6 PM and 8 PM IST.')

Choropleth map is only available between 6 PM and 8 PM IST.


In [348]:
# task 3 :Plot a time series line chart to show the trend of total installs over time.
# adding filter 
filter_time_df = merged_df[(merged_df['Reviews'] > 500) & (~merged_df['App'].str.startswith(('X','Y','Z'), na=False)) & (~merged_df['App'].str.contains('S', case=False, na=False)) & (merged_df['Category'].str.startswith(('E','C','B'), na=False))].copy()
# category Translation
category_translation_2 = {
    "BEAUTY": "सौंदर्य",
    "BUSINESS": "வணிகம்",
    "DATING": "Verabredung",
} 
filter_time_df['Category'] = filter_time_df['Category'].replace(category_translation_2)
#month column
filter_time_df['Last Updated'] = pd.to_datetime(filter_time_df['Last Updated'], errors='coerce')
filter_time_df['Month'] = filter_time_df['Last Updated'].dt.to_period('M').astype(str)
# monthly installs
monthly_installs = filter_time_df.groupby(['Month','Category'])['Installs'].sum().reset_index()
monthly_installs = monthly_installs.sort_values('Month')
# growth percentage
monthly_installs['Growth'] = monthly_installs.groupby('Category')['Installs'].pct_change() 
# time restriction
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist)
if 18 <= current_time.hour < 21:
    show_graph = True
else:
    show_graph = False
if show_graph and not monthly_installs.empty:
    fig13 = px.line(
        monthly_installs,
        x='Month',
        y='Installs',
        color='Category',
        title='Monthly Installs Trend by Category',
        color_discrete_sequence=px.colors.qualitative.Set1,
        width=plot_width,
        height=plot_height
    )
    # highlight growth > 20%
    growth_data = monthly_installs[monthly_installs["Growth"] > 0.20]
    for cat in growth_data['Category'].unique():
        temp = growth_data[growth_data['Category'] == cat]
        fig13.add_scatter(
            x=growth_data['Month'],
            y=growth_data['Installs'],
            fill='tozeroy',
            mode='none',
            opacity=0.2
)
    fig13.update_layout(
        plot_bgcolor=plot_bg_color,
        paper_bgcolor=plot_bg_color,
        font_color=text_color,
        title_font=title_font,
        xaxis=dict(title_font=axis_font),
        yaxis=dict(title_font=axis_font),
        margin=dict(l=10,r=10,t=30,b=10)
    )
    fig13.update_traces(marker=dict(line=dict(color='white',width=1)))
    save_plot_as_html(fig13, "Monthly Installs Graph 13.html", "This line chart shows the trend of total installs over time for different categories. It helps identify seasonal patterns and growth trends.")
    fig13.show()
else:
    print('Time series line chart is only available between 6 PM and 9 PM IST.')

In [350]:
# Task 4 :create a stacked area chart to visualize the cumulative number of installs over time for each app category.
#adding filter
filter_area_df = merged_df[(merged_df['Rating'] >= 4.2) & (~merged_df['App'].str.contains(r'\d', na=False)) & (merged_df['Category'].str.startswith(('T','P'), na = False)) & (merged_df['Reviews'] > 1000) & (merged_df['Size'] >= 20) & (merged_df['Size'] <= 80)].copy()
category_translation_3 = {
    "TRAVEL_AND_LOCAL": "Voyage et Local",
    "PRODUCTIVITY" : "Productividad",
    "PHOTOGRAPHY" : "写真"
}
filter_area_df['Category'] = filter_area_df['Category'].replace(category_translation_3)
# time series 
filter_area_df['Last Updated'] = pd.to_datetime(filter_area_df['Last Updated'], errors='coerce')
filter_area_df['Month'] = filter_area_df['Last Updated'].dt.to_period('M').astype(str)
# monthly_installs
monthly_installs_area = filter_area_df.groupby(['Month','Category'])['Installs'].sum().reset_index()
monthly_installs_area = monthly_installs_area.sort_values(['Category','Month'])
monthly_installs_area['Growth'] = monthly_installs_area.groupby('Category')['Installs'].pct_change()
# Cumulative_Installs
monthly_installs_area['Cumulative_Installs'] = monthly_installs_area.groupby('Category')['Installs'].cumsum()
# time restriction
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist)
if 16 <= current_time.hour < 18:
    show_graph = True
else:
    show_graph = False
if show_graph and not monthly_installs_area.empty:
    # Highlight growth >25% by increasing opacity
     monthly_installs_area["Opacity"] = monthly_installs_area["Growth"].apply(lambda x: 1 if x > 0.25 else 0.5)
     fig14 = px.area(
         monthly_installs_area,
         x="Month",
         y="Cumulative_Installs",
         color="Category",
         title="Cumulative Installs Over Time by App Category",
         opacity=monthly_installs_area["Opacity"]
)
     fig14.update_layout(
         xaxis_title="Month",
         yaxis_title="Cumulative Installs",
         template="plotly_dark"
    )
     fig14.show()
     save_plot_as_html(fig14, "Cumulative Installs Graph 14.html", "This stacked area chart visualizes the cumulative number of installs over time for each app category, highlighting periods of significant growth.")
else:
    print("Stacked area chart is available only between 4 PM and 6 PM IST.")


Stacked area chart is available only between 4 PM and 6 PM IST.


In [336]:
# task 5 : Creating grouped bar chart to compare the average rating and total review count for the top 10 app categories by number of installs
filter_bar_df = merged_df[(merged_df['Rating'] >= 4.0) & (merged_df['Size'] >= 10)].copy()
#January filter
filter_bar_df['Last Updated'] = pd.to_datetime(filter_bar_df['Last Updated'], errors='coerce')
filter_bar_df = filter_bar_df[filter_bar_df['Last Updated'].dt.month == 1]
# Top 10 Categories by installs
Top_categories = (filter_bar_df.groupby('Category')['Installs'].sum().sort_values(ascending=False).head(10).index)
filter_bar_df = filter_bar_df[filter_bar_df['Category'].isin(Top_categories)]
#Rating and Review Matrics
Category_stats = (filter_bar_df.groupby('Category').agg({"Rating": "mean", "Reviews": "sum"})).reset_index()
Category_stats.rename(columns={"Rating" : "Average Rating","Reviews" : "Total Reviews"},inplace= True)
Category_stats = Category_stats.sort_values("Total Reviews", ascending=False)
# time restriction
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist)
if 15 <= current_time.hour < 17:
    show_graph = True
else:
    show_graph = False
if show_graph and not Category_stats.empty:
    fig15 = px.bar(
        Category_stats,
        x='Category',
        y=["Average Rating", "Total Reviews"],
        barmode='group',
        title="Average Rating and Total Reviews by top 10 Category",
    )
    fig15.update_layout(
        template="plotly_dark",
        xaxis_title="App Category",
        yaxis_title='Value'
    )
    fig15.show()
    save_plot_as_html(fig15, "Rating Reviews Graph 15.html", "This grouped bar chart compares the average rating and total review count for the top 10 app categories by number of installs, providing insights into user engagement and satisfaction.")
else:
    print("Grouped bar chart is available only between 3 PM and 5 PM IST.")

Grouped bar chart is available only between 3 PM and 5 PM IST.


In [337]:
# Task 6:Create a dual-axis chart comparing the average installs and revenue for free vs. paid apps within the top 3 app categories.
# data cleaning
# android ver to numeric
merged_df['Android Ver'] = merged_df['Android Ver'].astype(str)
merged_df['Android Ver'] = (merged_df['Android Ver'].str.extract(r'(\d+\.?\d*)').astype(float))
# adding filter
filter_axis_df = merged_df[(merged_df['Installs'] >= 10000) & (merged_df['Revenue'] >= 10000) & (merged_df['Android Ver'] > 4.0) & (merged_df['Size'] > 15) & (merged_df['Content Rating'] == 'Everyone') & (merged_df['App'].str.len() <= 30)].copy()
Top_categories =(filter_axis_df.groupby('Category')['Installs'].sum().sort_values(ascending=False).head(3).index)
filter_axis_df = filter_axis_df[filter_axis_df['Category'].isin(Top_categories)]
# Free an paid app
Category_stats = (filter_axis_df.groupby('Type').agg({'Installs':'mean','Revenue' : 'sum'}).reset_index())
Category_stats.rename(columns={'Installs':'Average Installs','Revenue':'Total Revenue'},inplace=True)
# time restriction
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist)
if 13 <= current_time.hour < 14:
    show_graph = True
else:
    show_graph = False
if show_graph and not Category_stats.empty:
     plot_df = Category_stats.melt(
        id_vars="Type",
        value_vars=["Average Installs", "Total Revenue"],
        var_name="Metric",
        value_name="Value"
    )
     fig16 = px.bar(
         plot_df,
         x="Type",
         y="Value",
         color="Metric",
         barmode="group",
         title="Average Installs vs Revenue (Free vs Paid Apps)"
    )
     fig16.update_layout(
        xaxis_title="App Type",
        yaxis_title="Value",
        template="plotly_dark"
    )
     fig16.show()
     save_plot_as_html(fig16, "Installs Revenue Graph 16.html", "This dual-axis chart compares the average installs and total revenue for free vs. paid apps within the top 3 app categories, highlighting differences in performance and monetization.")
else:
    print("This chart is available only between 1 PM and 2 PM IST.")

This chart is available only between 1 PM and 2 PM IST.


In [338]:
plot_containers_split=plot_containers.split('</div>')

In [339]:
if len(plot_containers_split) > 1:
    final_plot=plot_containers_split[-2]+'</div>'
else:
    final_plot=plot_containers

In [349]:
# HTML template for the dashboard
dashboard_html = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Google Play Store Reviews Analytics</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            background-color: #333;
            color: #fff;
            margin: 0;
            padding: 0;
        }}
        .header {{
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 20px;
            background-color: #444;
        }}
        .header img {{
            margin: 0 10px;
            height: 50px;
        }}
        .container {{
            display: flex;
            flex-wrap: wrap;
            justify-content: center;
            padding: 20px;
        }}
        .plot-container {{
            border: 2px solid #555;
            margin: 10px;
            padding: 10px;
            width: {plot_width}px;
            height: {plot_height}px;
            overflow: hidden;
            position: relative;
            cursor: pointer;
        }}
        .insights {{
            display: none;
            position: absolute;
            right: 10px;
            top: 10px;
            background-color: rgba(0, 0, 0, 0.7);
            padding: 5px;
            border-radius: 5px;
            color: #fff;
        }}
        .plot-container:hover .insights {{
            display: block;
        }}
    </style>
    <script>
        function openPlot(filename) {{
            window.open(filename, '_blank');
        }}
    </script>
</head>
<body>
    <div class="header">
        <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/4/4a/Logo_2013_Google.png/800px-Logo_2013_Google.png" alt="Google Logo">
        <h1>Google Play Store Reviews Analytics</h1>
        <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/7/78/Google_Play_Store_badge_EN.svg/1024px-Google_Play_Store_badge_EN.svg.png" alt="Google Play Store Logo">
    </div>
    <div class="container">
        {plots}
    </div>
</body>
</html>
"""


In [341]:
final_html=dashboard_html.format(plots=plot_containers,plot_width=plot_width,plot_height=plot_height)

In [342]:

dashboard_path=os.path.join(html_files_path,"dashboard.html")

In [343]:
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(final_html)

In [344]:
webbrowser.open('file://'+os.path.realpath(dashboard_path))

True